### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    "groq:openai/gpt-oss-20b"
)

response = model.invoke("Why do parrots talk?")
response

AIMessage(content='### In a nutshell  \nParrots don’t “talk” the way humans do—they’re **improvisational vocalists**.  \nTheir brains, vocal anatomy, and social instincts make them superb imitators of the sounds they hear, and when they’re kept as pets, the most familiar “sound” they hear is a human voice.  \n\n---\n\n## 1. The anatomy of a parrot’s voice\n\n| Feature | Why it matters |\n|---------|----------------|\n| **Syrinx** (the bird’s “voice box”) | Located at the base of the trachea, it can produce two independent sounds at once, giving parrots a wide pitch range and the ability to mimic complex phrases. |\n| **Highly mobile tongue** | Allows fine control over vowel sounds. |\n| **Large brain‑to‑body ratio** | Especially in species like the African Grey, the parrot’s brain has a large **telencephalon** (the part that handles learning and memory). |\n| **Auditory cortex** | Parrots have a sophisticated auditory processing system that lets them recognize and reproduce specific so

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_e0e9fcdd-829c-46d0-b0a0-9cdc4fd372d3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 127, 'total_tokens': 163, 'completion_time': 0.03691552, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006183848, 'prompt_tokens_details': None, 'queue_time': 0.390814707, 'total_time': 0.043099368}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d23c14756c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a12bc4-8b76-7bf0-a86e-a58625257bff-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_e0e9fcdd-829c-46d0-b0a0-9cdc4fd372d3', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_token

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It’s sunny in Boston!


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather in Boston?" We need to use the provided function get_weather with location "Boston". Then respond with the result.', 'tool_calls': [{'id': 'fc_a7115af4-c248-4e9f-b7c0-21bbb588357c', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 126, 'total_tokens': 181, 'completion_time': 0.060965128, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.007202864, 'prompt_tokens_details': None, 'queue_time': 0.329132821, 'total_time': 0.068167992}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_34cb00738d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12bc4-ea66-7f93-a60b-8719d863f6a8-0', tool_calls=[{'name':